# Project 1: Ensemble of Trained DNNs

This notebook combines several already trained networks into **one Keras model** that averages their predictions of $\log_2$ m-height. The project explicitly allows several DNNs ("mixture of experts"). Averaging uses a standard `Average` layer, so the combined model is still made only of neural-network layers.

**How the members are chosen.** Each candidate network is evaluated on the provided-data **validation** rows. All combinations of up to four candidates are then scored there, and the best combination is kept. The held-out test rows are only reported, never used for the choice. All candidates were trained with the same seeded split, so these validation and test rows were never used to train any of them.

The members take different inputs: the MLPs read the padded `P` (`matrix`, 6 × 5) and the transformer reads $G = [I_k \mid P]$ (`generator`, 6 × 9). Both also read `[n, k, m]`. The combined model takes all three, and the final cell builds them from the same test samples.


In [ ]:
# Install dependencies (Google Colab only; local/NERSC runs use the project environment).

import sys

if "google.colab" in sys.modules:
    %pip install -q "numpy>=2,<3" "tensorflow==2.21.0" "keras==3.15.1"


## Configuration

| Setting | Default | Environment variable |
|---|---|---|
| Candidate runs (folders in `artifacts/`) | V2, V4 run A, V3 random-weight and V3 pretrained | `MHEIGHT_CANDIDATES` (comma-separated) |
| Members | `auto`: best validation combination | `MHEIGHT_MEMBERS` (comma-separated, overrides `auto`) |
| Output directory | `artifacts/ensemble` | `MHEIGHT_RUN` |


In [ ]:
import itertools
import json
import os
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

SEED = 636
CANDIDATES = os.environ.get(
    "MHEIGHT_CANDIDATES",
    "dnn_v2_finetune,dnn_v4_A_plain_data,dnn_v3_bert-small_scratch,dnn_v3_bert-small_pretrained").split(",")
MEMBERS = os.environ.get("MHEIGHT_MEMBERS", "auto")
MAX_MEMBERS = 4
ARTIFACT_DIR = Path("artifacts") / os.environ.get("MHEIGHT_RUN", "ensemble")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = ARTIFACT_DIR / "best_model.keras"
PARAMETER_SETTINGS = [(9, k, m) for k in (4, 5, 6) for m in range(2, 9 - k + 1)]
EVAL_DEVICE = "/GPU:0" if tf.config.list_physical_devices("GPU") else "/CPU:0"

tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow:", tf.__version__, "| Keras:", tf.keras.__version__, "| evaluation device:", EVAL_DEVICE)
print("Candidates:", CANDIDATES)


## Load the Provided Data and the Shared Split

Only the provided data is needed: the candidates are compared on its validation and test rows. The split is read from a candidate's `split_indices.npz`. The extended files start with the provided rows, so indices below `N_PROVIDED` refer to the same rows here. The check below confirms that every candidate used the same split.


In [ ]:
NUMPY_MAJOR = int(np.__version__.split(".")[0])


class NumpyCompatUnpickler(pickle.Unpickler):
    # NumPy 2 pickles reference numpy._core; NumPy 1.x only provides numpy.core.
    def find_class(self, module, name):
        if NUMPY_MAJOR < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as file:
        return NumpyCompatUnpickler(file).load()


def pack_inputs(samples):
    """All three member inputs: padded P, padded G = [I_k | P], and [n, k, m]."""
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.zeros((len(samples), 3), dtype=np.float32)
    for row, (n, k, m, P) in enumerate(samples):
        P = np.asarray(P, dtype=np.float64)
        if P.shape != (k, n - k):
            raise ValueError(f"Sample {row}: P has shape {P.shape}, expected {(k, n - k)}")
        matrices[row, :k, :n - k] = P
        generators[row, :k, :k] = np.eye(k)
        generators[row, :k, k:n] = P
        parameters[row] = (n, k, m)
    return {"matrix": matrices, "generator": generators, "parameters": parameters}


features = load_pickle("data/CSCE-636-Project-1-Train-n_k_m_P")
labels = np.asarray(load_pickle("data/CSCE-636-Project-1-Train-mHeights"), dtype=np.float64)
N_PROVIDED = len(features)
splits = {}
for run in CANDIDATES:
    saved = np.load(Path("artifacts") / run / "split_indices.npz")
    current = {name: saved[name][saved[name] < N_PROVIDED] for name in ("validation", "test")}
    if splits and any(not np.array_equal(current[name], splits[name]) for name in current):
        raise ValueError(f"{run} used a different split")
    splits = current
inputs = pack_inputs(features)
setting_ids = np.array([PARAMETER_SETTINGS.index((n, k, m)) for n, k, m, _ in features], dtype=np.int8)
log2_labels = np.log2(labels)
print(f"Provided rows: {N_PROVIDED:,}; validation {len(splits['validation']):,}; test {len(splits['test']):,}")


## Evaluate Each Candidate

Each candidate's saved model predicts the validation and test rows. Predictions are $\log_2$ m-heights, so averaging them is averaging in the log domain, the domain of the grading cost. The cost is the mean over the 9 settings of $\sigma_{n,k,m}$.


In [ ]:
def load_member(run, device=EVAL_DEVICE):
    with tf.device(device):
        return tf.keras.models.load_model(str(Path("artifacts") / run / "best_model.keras"), compile=False)


def member_inputs(network, batch):
    return {tensor.name: batch[tensor.name] for tensor in network.inputs}


def predict_log2(network, indices, device=EVAL_DEVICE, batch_size=8192):
    @tf.function(jit_compile=True)
    def forward(batch):
        return network(batch, training=False)

    out = np.empty(len(indices), dtype=np.float64)
    with tf.device(device):
        for start in range(0, len(indices), batch_size):
            selected = indices[start:start + batch_size]
            batch = member_inputs(network, {name: values[selected] for name, values in inputs.items()})
            out[start:start + len(selected)] = np.asarray(forward(batch)).reshape(-1)
    return out


def setting_costs(indices, predicted):
    errors = (log2_labels[indices] - predicted) ** 2
    return np.array([errors[setting_ids[indices] == g].mean() for g in range(len(PARAMETER_SETTINGS))])


predictions = {}
for run in CANDIDATES:
    network = load_member(run)
    predictions[run] = {name: predict_log2(network, idx) for name, idx in splits.items()}
    val, test = (setting_costs(splits[s], predictions[run][s]) for s in ("validation", "test"))
    print(f"{run:32s} validation {val.mean():.4f} | test {test.mean():.4f} | test (9,4,5) {test[3]:.3f} (9,5,4) {test[6]:.3f} (9,6,3) {test[8]:.3f}")


## Choose the Members on the Validation Rows

All combinations of 1 to 4 candidates are scored by averaging their validation predictions. The lowest validation cost wins; on a tie, fewer members win. Test costs are shown only for reference.


In [ ]:
results = []
for size in range(1, min(MAX_MEMBERS, len(CANDIDATES)) + 1):
    for combo in itertools.combinations(CANDIDATES, size):
        val = setting_costs(splits["validation"], np.mean([predictions[r]["validation"] for r in combo], axis=0)).mean()
        test = setting_costs(splits["test"], np.mean([predictions[r]["test"] for r in combo], axis=0)).mean()
        results.append((val, size, combo, test))
results.sort()
for val, size, combo, test in results[:8]:
    print(f"validation {val:.4f} | test {test:.4f} | {' + '.join(combo)}")
members = list(results[0][2]) if MEMBERS == "auto" else MEMBERS.split(",")
print("\nMembers:", members)


## Build and Save the Combined Model

Each member is loaded as a sub-model and receives the inputs it was trained with. The members' outputs ($\log_2$ m-heights, each already bounded to `[0, 100]`) are averaged by an `Average` layer, so the result stays in `[0, 100]`. Every prediction $2^z$ is therefore finite and at least 1. The combined model is saved as one `.keras` file; reloading it reproduces the same predictions.


In [ ]:
def build_ensemble(runs):
    model_inputs = {
        "matrix": tf.keras.Input(shape=(6, 5), name="matrix"),
        "generator": tf.keras.Input(shape=(6, 9), name="generator"),
        "parameters": tf.keras.Input(shape=(3,), name="parameters"),
    }
    outputs = []
    for index, run in enumerate(runs):
        member = load_member(run, "/CPU:0")
        member.name = f"member_{index}"
        outputs.append(member({tensor.name: model_inputs[tensor.name] for tensor in member.inputs}))
    output = tf.keras.layers.Average(name="log2_height")(outputs) if len(outputs) > 1 else outputs[0]
    return tf.keras.Model(inputs=model_inputs, outputs=output, name="mheight_ensemble")


ensemble = build_ensemble(members)
ensemble.save(MODEL_PATH)
(ARTIFACT_DIR / "members.json").write_text(json.dumps({"members": members}, indent=2) + "\n")
reloaded = tf.keras.models.load_model(str(MODEL_PATH), compile=False)
check = {name: values[splits["test"][:2048]] for name, values in inputs.items()}
if not np.array_equal(np.asarray(ensemble(check, training=False)), np.asarray(reloaded(check, training=False))):
    raise RuntimeError("The reloaded ensemble does not reproduce the saved model")
print(f"Saved {MODEL_PATH} ({reloaded.count_params():,} parameters) with members {members}")


## Test the Ensemble

The combined model is evaluated on the provided-data validation and test rows (on the GPU when available). The σ check then repeats the test-row predictions **on the CPU with fixed batches of 1024**, the path used by the final cell, and saves and reloads them as a pickle list.


In [ ]:
ensemble_model = load_member(ARTIFACT_DIR.name)
cost_report = {}
for split in ("validation", "test"):
    costs = setting_costs(splits[split], predict_log2(ensemble_model, splits[split]))
    cost_report[split] = {"macro_sigma": float(costs.mean()),
                          "per_setting": {str(s): float(c) for s, c in zip(PARAMETER_SETTINGS, costs)}}
    print(f"Ensemble {split}: mean over settings {costs.mean():.6f}")
    for setting, cost in zip(PARAMETER_SETTINGS, costs):
        print(f"  {setting}: {cost:.6f}")
(ARTIFACT_DIR / "costs.json").write_text(json.dumps({"members": members, **cost_report}, indent=2) + "\n")

cpu_model = load_member(ARTIFACT_DIR.name, "/CPU:0")
cpu_log2 = predict_log2(cpu_model, splits["test"], device="/CPU:0", batch_size=1024)
prediction_path = ARTIFACT_DIR / "local_test_predictions.pkl"
with prediction_path.open("wb") as file:
    pickle.dump(np.exp2(cpu_log2).tolist(), file)
with prediction_path.open("rb") as file:
    reread = np.log2(np.asarray(pickle.load(file)))
cpu_costs = setting_costs(splits["test"], reread)
print(f"CPU path (as in the final cell), test: mean over settings {cpu_costs.mean():.6f}")


## Final Cell: Predict m-Heights for a Test File

This is the cell for testing the submitted model. It is **self-contained**: in a fresh session, run the first cell (it installs the pinned TensorFlow/Keras versions on Colab), then run this cell. None of the training cells or the training data are needed.

1. Place the submitted model file `best_model.keras` next to this notebook. When run from the project folder, the cell also finds it at `artifacts/ensemble/best_model.keras`.
2. Set `TEST_INPUT_PATH` to the test file: a pickled list of samples `[n, k, m, P]`, in the same format as `CSCE-636-Project-1-Train-n_k_m_P`.
3. Run the cell. It writes `OUTPUT_PATH`: a pickled list of predicted m-heights (Python floats), one per input sample in the same order, in the same format as `CSCE-636-Project-1-Train-mHeights`.

The processing is the same as in the evaluation above:
- `P` is placed in a zero-padded `6 × 5` array and $G = [I_k \mid P]$ in a zero-padded `6 × 9` array; `[n, k, m]` is passed alongside them.
- The ensemble outputs the average $z = \log_2 \hat y$ of its members, in `[0, 100]`, and the only post-processing is $\hat y = 2^z$.

Every prediction is therefore finite and at least 1. The cell checks this before saving. Inference runs on the CPU in fixed batches of 1024, so repeated runs give identical results.


In [ ]:
# ===== Final cell: load the trained DNN and predict m-heights for a test file =====
import pickle
from pathlib import Path

import numpy as np
import tensorflow as tf

TEST_INPUT_PATH = "CSCE-636-Project-1-Test-n_k_m_P"   # pickled list of [n, k, m, P]
OUTPUT_PATH = "CSCE-636-Project-1-Test-mHeights"      # pickled list of predicted m-heights
MODEL_CANDIDATES = ["best_model.keras", "artifacts/ensemble/best_model.keras"]
BATCH = 1024


class CompatUnpickler(pickle.Unpickler):
    # Pickles written with NumPy 2 reference numpy._core; NumPy 1.x only has numpy.core.
    def find_class(self, module, name):
        if int(np.__version__.split(".")[0]) < 2 and module.startswith("numpy._core"):
            module = "numpy.core" + module[len("numpy._core"):]
        return super().find_class(module, name)


if not Path(TEST_INPUT_PATH).exists():
    # Nothing to predict yet (e.g. a batch run before the test file is available).
    print(f"Test file {TEST_INPUT_PATH!r} not found: set TEST_INPUT_PATH and run this cell again.")
else:
    model_file = next((path for path in MODEL_CANDIDATES if Path(path).exists()), None)
    if model_file is None:
        raise FileNotFoundError(f"Trained model not found; looked for {MODEL_CANDIDATES}")
    with tf.device("/CPU:0"):
        network = tf.keras.models.load_model(model_file, compile=False)


    @tf.function(jit_compile=True)
    def predict_log2(batch):
        # XLA-compiled forward pass: several times faster on CPU, deterministic for a given machine.
        return network(batch, training=False)

    with open(TEST_INPUT_PATH, "rb") as file:
        samples = CompatUnpickler(file).load()

    # Inputs: P padded to 6 x 5, G = [I_k | P] padded to 6 x 9, and [n, k, m].
    matrices = np.zeros((len(samples), 6, 5), dtype=np.float32)
    generators = np.zeros((len(samples), 6, 9), dtype=np.float32)
    parameters = np.zeros((len(samples), 3), dtype=np.float32)
    for row, (n, k, m, P) in enumerate(samples):
        P = np.asarray(P, dtype=np.float64)
        if P.shape != (k, n - k):
            raise ValueError(f"Sample {row}: P has shape {P.shape}, expected {(k, n - k)}")
        matrices[row, :k, :n - k] = P
        generators[row, :k, :k] = np.eye(k)
        generators[row, :k, k:n] = P
        parameters[row] = (n, k, m)

    # The network outputs z = log2(m-height); the only post-processing is 2**z.
    log2_heights = np.empty(len(samples), dtype=np.float64)
    with tf.device("/CPU:0"):
        for start in range(0, len(samples), BATCH):
            batch = {"matrix": matrices[start:start + BATCH],
                     "generator": generators[start:start + BATCH],
                     "parameters": parameters[start:start + BATCH]}
            log2_heights[start:start + BATCH] = np.asarray(predict_log2(batch)).reshape(-1)
    predicted_heights = np.exp2(log2_heights)

    if len(predicted_heights) != len(samples) or not np.isfinite(predicted_heights).all() \
            or np.any(predicted_heights < 1):
        raise ValueError("Every prediction must be finite and at least 1; nothing was saved")
    with open(OUTPUT_PATH, "wb") as file:
        pickle.dump(predicted_heights.tolist(), file)

    print(f"Model: {model_file} (TensorFlow {tf.__version__}, Keras {tf.keras.__version__})")
    print(f"Predicted {len(predicted_heights):,} m-heights -> {OUTPUT_PATH}")
    print(f"Range: {predicted_heights.min():.4f} to {predicted_heights.max():.4f}; first five: {predicted_heights[:5].tolist()}")
